# 03. Feature Engineering

Objetivo: Limpieza de variables colineales, reversion de variables dummies categóricas y creación de métricas de engagement de alto valor (PMP, OAS, PLI, EROI).

In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def drop_redundant_columns(df: pd.DataFrame, prefix_to_drop: str = 'N_CLM') -> pd.DataFrame:
    """
    Removes redundant columns (e.g. Medicare claims) that introduce collinearity.
    """
    cols = [c for c in df.columns if c.startswith(prefix_to_drop)]
    logger.info(f"Dropping {len(cols)} redundant columns with prefix {prefix_to_drop}.")
    return df.drop(columns=cols)

def reverse_dummies(df: pd.DataFrame, prefix: str, target_name: str) -> pd.DataFrame:
    """
    Reverses categorical dummy variables (e.g., SPEC_GE_, STATE_1_) into a single categorical column.
    """
    dummy_cols = [c for c in df.columns if c.startswith(prefix)]
    if not dummy_cols:
        return df
        
    logger.info(f"Reversing dummy features for {prefix} into column {target_name}.")
    df_out = df.copy()
    
    # Extracting original category name by index of maximum truth value
    df_out[target_name] = df_out[dummy_cols].idxmax(axis=1).apply(lambda x: str(x).replace(prefix, ''))
    df_out = df_out.drop(columns=dummy_cols)
    return df_out


## Business Domain Ratios

In [ ]:
def create_business_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates core business KPIs specific to pharmaceutical strategy.
    Uses epsilon injection to safely handle unengaged HCPs preventing ZeroDivisionErrors.
    """
    epsilon = 1e-5
    df_out = df.copy()
    logger.info("Computing pharmaceutical business ratios.")
    
    # Pfizer Market Penetration
    if 'BRAND1_NBRX' in df_out.columns and 'UC_NBRX' in df_out.columns:
        df_out['pfizer_market_penetration'] = df_out['BRAND1_NBRX'] / (df_out['UC_NBRX'] + epsilon)
        
    # Oral Adoption Speed
    if 'ORAL_NBRX' in df_out.columns and 'UC_NBRX' in df_out.columns:
        df_out['oral_adoption_speed'] = df_out['ORAL_NBRX'] / (df_out['UC_NBRX'] + epsilon)
        
    # Pfizer Loyalty Index
    if 'BRAND1_NBRX' in df_out.columns and 'UC_TRX' in df_out.columns:
        df_out['pfizer_loyalty_index'] = df_out['BRAND1_NBRX'] / (df_out['UC_TRX'] + epsilon)
        
    # Engagement ROI
    if 'BRAND1_NBRX' in df_out.columns and 'DETAILS' in df_out.columns and 'SAMPLES' in df_out.columns:
        rte = df_out['RTE'] if 'RTE' in df_out.columns else 0
        total_engagement = df_out['DETAILS'] + df_out['SAMPLES'] + rte
        df_out['engagement_roi'] = df_out['BRAND1_NBRX'] / (total_engagement + epsilon)
        
    return df_out
